#### Static

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import dgl
import dgl.nn as dglnn
from sklearn.preprocessing import LabelEncoder, StandardScaler

# -------------------------
# Load and Prepare Dataset
# -------------------------
df = pd.read_csv("../all_circuits_features.csv")

label_encoder = LabelEncoder()
df["gate_label"] = label_encoder.fit_transform(df["gate_type"])

feature_columns = [
    "fan_in", "fan_out", "dist_to_output", "is_primary_input", "is_primary_output",
    "is_internal", "is_key_gate", "degree_centrality", "betweenness_centrality",
    "closeness_centrality", "clustering_coefficient", "avg_fan_in_neighbors", "avg_fan_out_neighbors"
]
df = df.dropna(subset=feature_columns)
df[feature_columns] = df[feature_columns].astype(float)
scaler = StandardScaler()
df[feature_columns] = scaler.fit_transform(df[feature_columns])

# -------------------------
# Build graph
# -------------------------
nodes = df["node"].tolist()
node_to_id = {node: i for i, node in enumerate(nodes)}
edges = []
for _, row in df.iterrows():
    node_id = node_to_id[row["node"]]
    potential_sources = df[df["fan_out"] > 0]["node"].tolist()
    k = int(row["fan_in"]) if not np.isnan(row["fan_in"]) else 0
    sources = potential_sources[:max(k, 0)]
    for src in sources:
        if src in node_to_id:
            edges.append((node_to_id[src], node_id))

src_nodes, dst_nodes = zip(*edges) if edges else ([], [])
graph = dgl.graph((torch.tensor(src_nodes), torch.tensor(dst_nodes)), num_nodes=len(nodes))
graph = dgl.add_self_loop(graph)
graph.ndata['features'] = torch.tensor(df[feature_columns].values, dtype=torch.float32)
graph.ndata['labels'] = torch.tensor(df["gate_label"].values, dtype=torch.long)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
graph = graph.to(device)
features = graph.ndata['features'].to(device)
labels = graph.ndata['labels'].to(device)

# -------------------------
# Class-incremental batches
# -------------------------
class_to_label = {c: i for i, c in enumerate(label_encoder.classes_)}
class_batches = [["and","not"],["nor"],["nand"],["input"],["output","or"],["xor"]]
all_labels_np = labels.cpu().numpy()
def indices_for_classes(class_names):
    class_ids = [class_to_label[c] for c in class_names]
    mask = np.isin(all_labels_np, class_ids)
    return np.where(mask)[0]
batch_indices = [indices_for_classes(b) for b in class_batches]

# -------------------------
# GraphSAGE model
# -------------------------
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden_feats, out_feats, dropout=0.5):
        super(GraphSAGE, self).__init__()
        self.layer1 = dglnn.SAGEConv(in_feats, hidden_feats, aggregator_type='mean')
        self.layer2 = dglnn.SAGEConv(hidden_feats, out_feats, aggregator_type='mean')
        self.dropout = nn.Dropout(dropout)
    def forward(self, g, inputs):
        x = self.layer1(g, inputs)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.layer2(g, x)
        return x

in_feats = features.shape[1]
out_feats = len(label_encoder.classes_)

# -------------------------
# Catastrophic Forgetting Metrics
# -------------------------
def weight_distance_metric(params_before, params_after):
    return sum(torch.norm(p_before - p_after, p=2).item()
               for p_before, p_after in zip(params_before, params_after))

def regularization_metric(params_before, params_after, fisher_matrix):
    return sum(torch.sum(F * (p_after - p_before) ** 2).item()
               for p_before, p_after, F in zip(params_before, params_after, fisher_matrix))

def replay_metric(acc_with_replay, acc_without_replay):
    return acc_with_replay - acc_without_replay

# -------------------------
# Training loop per mechanism
# -------------------------
def run_mechanism(mech, config):
    model = GraphSAGE(in_feats, config["hidden_feats"], out_feats, dropout=config["dropout"]).to(device)
    opt = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()

    params_before = [p.detach().clone() for p in model.parameters()]
    batch_accs = []

    for b, idx in enumerate(batch_indices):
        print(b)
        train_idx_t = torch.tensor(idx, dtype=torch.long, device=device)
        for epoch in range(config["epochs_per_batch"]):
            model.train()
            logits = model(graph, features)
            loss = loss_fn(logits[train_idx_t], labels[train_idx_t])
            opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            logits = model(graph, features)
            preds = logits.argmax(dim=1)
            acc = (preds[train_idx_t] == labels[train_idx_t]).float().mean().item()
        batch_accs.append(acc)

    params_after = [p.detach().clone() for p in model.parameters()]
    wd = weight_distance_metric(params_before, params_after)
    fisher_matrix = [torch.ones_like(p) for p in params_before]
    reg = regularization_metric(params_before, params_after, fisher_matrix)
    replay_eff = None
    if mech in ["rehearsal","gen_replay"]:
        replay_eff = replay_metric(batch_accs[-1], batch_accs[0])

    return {"mechanism": mech,
            "batch_accs": batch_accs,
            "weight_distance": wd,
            "regularization": reg,
            "replay_effect": replay_eff}

# -------------------------
# Best hyperparameters per mechanism (from your table)
# -------------------------
BEST_CONFIGS = {
    "naive":      {"hidden_feats":128,"dropout":0.0,"lr":0.0005,"weight_decay":0.0,"epochs_per_batch":27},
    "rehearsal":  {"hidden_feats":256,"dropout":0.3,"lr":0.0005,"weight_decay":0.0,"epochs_per_batch":15},
    "ewc":        {"hidden_feats":256,"dropout":0.0,"lr":0.0010,"weight_decay":0.0,"epochs_per_batch":9},
    "si":         {"hidden_feats":256,"dropout":0.0,"lr":0.0005,"weight_decay":0.0,"epochs_per_batch":15},
    "gen_replay": {"hidden_feats":128,"dropout":0.0,"lr":0.0010,"weight_decay":0.0,"epochs_per_batch":27},
}

# -------------------------
# Run all mechanisms
# -------------------------
results = []
for mech, cfg in BEST_CONFIGS.items():
    print(mech)
    res = run_mechanism(mech, cfg)
    results.append(res)

# -------------------------
# Print results
# -------------------------
for r in results:
    print(f"\nMechanism: {r['mechanism']}")
    print(f"Batch Accuracies: {r['batch_accs']}")
    print(f"Weight Distance Metric: {r['weight_distance']:.4f}")
    print(f"Regularization Metric: {r['regularization']:.4f}")
    if r['replay_effect'] is not None:
        print(f"Replay Effectiveness Metric: {r['replay_effect']:.4f}")


naive
0
1
2
3
4
5
rehearsal
0
1
2
3
4
5
ewc
0
1
2
3
4
5
si
0
1
2
3
4
5
gen_replay
0
1
2
3
4
5

Mechanism: naive
Batch Accuracies: [0.8152173757553101, 0.9629629850387573, 0.9992349147796631, 0.9949643015861511, 0.5883399844169617, 1.0]
Weight Distance Metric: 4.6314
Regularization Metric: 4.7124

Mechanism: rehearsal
Batch Accuracies: [0.7285770773887634, 0.4091220796108246, 0.2084927260875702, 0.9949643015861511, 0.37561193108558655, 0.9180327653884888]
Weight Distance Metric: 3.9276
Regularization Metric: 3.4348
Replay Effectiveness Metric: 0.1895

Mechanism: ewc
Batch Accuracies: [0.8754545450210571, 0.05384087935090065, 0.0, 0.9932857751846313, 0.5874499082565308, 1.0]
Weight Distance Metric: 4.3799
Regularization Metric: 4.3387

Mechanism: si
Batch Accuracies: [0.8877272605895996, 0.9873113632202148, 0.5818668603897095, 0.9932857751846313, 0.5883399844169617, 0.17213115096092224]
Weight Distance Metric: 3.1988
Regularization Metric: 2.3210

Mechanism: gen_replay
Batch Accuracies: 

#### Dynamic

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import dgl.nn as dglnn

# -------------------------
# Helpers (fresh definitions)
# -------------------------
def _to_list(idx):
    if isinstance(idx, np.ndarray): return idx.tolist()
    if isinstance(idx, list): return idx
    return list(idx)

class RehearsalBuffer:
    def __init__(self, size=10000, num_nodes=None):
        self.size = size
        self.data_idx = []
        self.num_nodes = num_nodes
    def add(self, idx):
        idx_list = _to_list(idx)
        self.data_idx.extend(idx_list)
        if self.num_nodes is not None:
            self.data_idx = [i for i in self.data_idx if 0 <= i < self.num_nodes]
        self.data_idx = list(dict.fromkeys(self.data_idx))
        if len(self.data_idx) > self.size:
            self.data_idx = np.random.choice(self.data_idx, self.size, replace=False).tolist()
    def get(self):
        return self.data_idx

# -------------------------
# Dynamic GraphSAGE model (expandable output head)
# -------------------------
class DynamicGraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden_feats, out_feats, dropout=0.0):
        super(DynamicGraphSAGE, self).__init__()
        self.sage1 = dglnn.SAGEConv(in_feats, hidden_feats, aggregator_type='mean')
        self.sage2 = dglnn.SAGEConv(hidden_feats, hidden_feats, aggregator_type='mean')
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(hidden_feats, out_feats)

    def forward(self, g, x):
        h = self.sage1(g, x)
        h = torch.relu(h)
        h = self.dropout(h)
        h = self.sage2(g, h)
        h = torch.relu(h)
        return self.out(h)

def expand_output_layer(model, new_out_feats, device):
    old_out = model.out
    old_out_feats = old_out.out_features
    if new_out_feats <= old_out_feats:
        return model, old_out_feats  # return old size for alignment
    in_features = old_out.in_features
    new_out = nn.Linear(in_features, new_out_feats, bias=True).to(device)
    with torch.no_grad():
        nn.init.normal_(new_out.weight, mean=0.0, std=0.01)
        nn.init.zeros_(new_out.bias)
        new_out.weight[:old_out_feats, :] = old_out.weight
        new_out.bias[:old_out_feats] = old_out.bias
    model.out = new_out
    return model, old_out_feats

# -------------------------
# Catastrophic forgetting metrics (shape-aligned overlap)
# -------------------------
def _aligned_overlap(a, b):
    # Slice both tensors to common shape (prefix overlap) to avoid expansion mismatch
    shape = tuple(min(x, y) for x, y in zip(a.shape, b.shape))
    slices = tuple(slice(0, s) for s in shape)
    return a[slices], b[slices]

def weight_distance_metric_aligned(params_before, params_after):
    total = 0.0
    for p0, p1 in zip(params_before, params_after):
        a, b = _aligned_overlap(p0, p1)
        total += torch.norm(a - b, p=2).item()
    return total

def regularization_metric_aligned(params_before, params_after, fisher_list):
    total = 0.0
    for p0, p1, F in zip(params_before, params_after, fisher_list):
        a, b = _aligned_overlap(p0, p1)
        Fa, _ = _aligned_overlap(F, F)  # just slice F to a's shape
        total += torch.sum(Fa * (b - a) ** 2).item()
    return total

def replay_metric(acc_last, acc_first):
    return acc_last - acc_first

# -------------------------
# Dynamic training loop per mechanism
# Relies on: graph, features, labels, class_batches, class_to_label, device defined in the static cell.
# -------------------------
in_feats = graph.ndata['features'].shape[1]  # recompute fresh

def run_dynamic_graphsage(mech, config):
    # Initialize to cover first batch classes
    first_ids = [class_to_label[c] for c in class_batches[0]]
    init_out_feats = max(first_ids) + 1
    model = DynamicGraphSAGE(in_feats, config["hidden_feats"], init_out_feats, dropout=config["dropout"]).to(device)
    opt = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()

    buffer = RehearsalBuffer(size=config.get("buffer_size", 0) or 10000,
                             num_nodes=graph.num_nodes()) if mech in ["rehearsal","gen_replay"] else None

    # Snapshot parameters before any training
    params_before = [p.detach().clone() for p in model.parameters()]
    batch_accs = []

    features = graph.ndata['features']
    labels = graph.ndata['labels']

    for b, idx in enumerate(batch_indices):
        # Classes seen so far and contiguous mapping
        seen_classes = [c for batch in class_batches[:b+1] for c in batch]
        seen_class_ids = [class_to_label[c] for c in seen_classes]
        target_out_feats = max(seen_class_ids) + 1

        # Expand output head if needed, reset optimizer
        if model.out.out_features < target_out_feats:
            model, old_rows = expand_output_layer(model, target_out_feats, device)
            opt = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

        # Build training indices (+ replay if any)
        train_idx = _to_list(idx)
        if buffer is not None:
            train_idx += buffer.get()
            train_idx = list(dict.fromkeys(train_idx))
        train_idx_t = torch.tensor(train_idx, dtype=torch.long, device=device)

        # Map global labels -> contiguous [0..C_seen-1]
        id_to_seen = {cid: i for i, cid in enumerate(seen_class_ids)}
        max_cid = max(seen_class_ids)
        map_tensor = torch.full((max_cid + 1,), -1, dtype=torch.long, device=device)
        for cid, j in id_to_seen.items():
            map_tensor[cid] = j
        y_global = labels[train_idx_t]
        assert torch.all(map_tensor[y_global] != -1), f"Labels outside seen set at batch {b+1}"
        y_seen = map_tensor[y_global]

        seen_cols_t = torch.tensor(seen_class_ids, dtype=torch.long, device=device)

        # Train
        for epoch in range(config["epochs_per_batch"]):
            model.train()
            logits_all = model(graph, features)
            logits_seen = logits_all[train_idx_t][:, seen_cols_t]
            loss = loss_fn(logits_seen, y_seen)
            opt.zero_grad(); loss.backward(); opt.step()

        # Update buffer
        if buffer is not None:
            buffer.add(idx)

        # Evaluate on training indices for seen slice
        with torch.no_grad():
            logits = model(graph, features)[:, seen_cols_t]
            preds = logits.argmax(dim=1)
            acc = (preds[train_idx_t] == y_seen).float().mean().item()
        batch_accs.append(acc)

    # Metrics with shape alignment (ignores newly added output rows)
    params_after = [p.detach().clone() for p in model.parameters()]
    wd = weight_distance_metric_aligned(params_before, params_after)
    fisher_matrix = [torch.ones_like(p) for p in params_before]  # simple importance proxy
    reg = regularization_metric_aligned(params_before, params_after, fisher_matrix)
    replay_eff = replay_metric(batch_accs[-1], batch_accs[0]) if mech in ["rehearsal","gen_replay"] else None

    return {
        "mechanism": mech,
        "batch_accs": batch_accs,
        "weight_distance": wd,
        "regularization": reg,
        "replay_effect": replay_eff
    }

# -------------------------
# Best hyperparameters per mechanism (GraphSAGE Dynamic)
# -------------------------
BEST_CONFIGS = {
    "naive":      {"hidden_feats":128, "dropout":0.0, "lr":0.0010, "weight_decay":0.0, "epochs_per_batch":6},
    "rehearsal":  {"hidden_feats":256, "dropout":0.0, "lr":0.0001, "weight_decay":0.0, "epochs_per_batch":120, "buffer_size":10000},
    "ewc":        {"hidden_feats":128, "dropout":0.5, "lr":0.0001, "weight_decay":0.0, "epochs_per_batch":18},
    "si":         {"hidden_feats":64,  "dropout":0.3, "lr":0.0010, "weight_decay":0.0, "epochs_per_batch":6},
    "gen_replay": {"hidden_feats":128, "dropout":0.0, "lr":0.0005, "weight_decay":0.0, "epochs_per_batch":54, "buffer_size":10000},
}

# -------------------------
# Run and print (second cell in the GraphSAGE notebook)
# -------------------------
results = []
for mech, cfg in BEST_CONFIGS.items():
    print(f"Running GraphSAGE dynamic: {mech}...")
    res = run_dynamic_graphsage(mech, cfg)
    results.append(res)

for r in results:
    print(f"\nMechanism: {r['mechanism']}")
    print(f"Batch Accuracies: {r['batch_accs']}")
    print(f"Weight Distance Metric (aligned): {r['weight_distance']:.4f}")
    print(f"Regularization Metric (aligned): {r['regularization']:.4f}")
    if r['replay_effect'] is not None:
        print(f"Replay Effectiveness Metric: {r['replay_effect']:.4f}")


Running GraphSAGE dynamic: naive...
Running GraphSAGE dynamic: rehearsal...
Running GraphSAGE dynamic: ewc...
Running GraphSAGE dynamic: si...
Running GraphSAGE dynamic: gen_replay...

Mechanism: naive
Batch Accuracies: [0.98687744140625, 1.0, 0.9980872273445129, 0.9970625042915344, 0.713840663433075, 1.0]
Weight Distance Metric (aligned): 4.4273
Regularization Metric (aligned): 4.6662

Mechanism: rehearsal
Batch Accuracies: [0.9995454549789429, 0.9539331197738647, 0.892817497253418, 0.9282887578010559, 0.9000571370124817, 0.9146413803100586]
Weight Distance Metric (aligned): 17.0837
Regularization Metric (aligned): 78.8594
Replay Effectiveness Metric: -0.0849

Mechanism: ewc
Batch Accuracies: [0.7042885422706604, 0.7355967164039612, 0.4311400055885315, 0.9723038077354431, 0.07076101750135422, 0.12295082211494446]
Weight Distance Metric (aligned): 1.5599
Regularization Metric (aligned): 0.5574

Mechanism: si
Batch Accuracies: [0.7098221182823181, 0.6858710646629333, 0.9605967998504639,